# Bloc 3A - Prevision des volumes (Prophet vs SARIMA vs XGBoost)

Ce notebook reproduit exactement la logique de `modelisation_volumes.py` du
pipeline PostgreSQL, pour evaluer **Prophet** sur un environnement ou il
fonctionne reellement (le binaire Stan precompile de Prophet plante sur
l'environnement Windows local du projet - probleme documente, independant
du code).

**Mode d'emploi : Menu "Execution" -> "Tout executer". Rien a modifier.**

A la fin, le notebook telecharge automatiquement 3 fichiers :
- `model_volumes.pkl` : le modele retenu (Prophet, SARIMA ou XGBoost, celui
  qui a le meilleur MAE sur le jeu de test)
- `predictions_volumes_j30.csv` : previsions J+1 a J+30 avec ce modele
- `metriques_bloc3a.txt` : rapport de comparaison des 3 modeles

Ramener ces 3 fichiers dans `mon_projet/colab_prophet/resultats/` pour
integration au pipeline local.

## 1. Installation et imports

In [ ]:
!pip install -q prophet xgboost statsmodels scikit-learn joblib


In [ ]:
import warnings
from datetime import timedelta
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor
from prophet import Prophet

warnings.filterwarnings("ignore")
print("Imports OK")


## 2. Chargement des donnees

Le fichier `features_journalier.csv` est le meme export que celui utilise
par le pipeline PostgreSQL local (table `features_journalier`). S'il n'est
pas deja present dans l'environnement Colab, la cellule suivante ouvre un
selecteur de fichier pour l'uploader (le fichier se trouve dans
`mon_projet/colab_prophet/features_journalier.csv` sur la machine locale).

In [ ]:
if not Path("features_journalier.csv").exists():
    from google.colab import files
    print("Merci d'uploader features_journalier.csv...")
    files.upload()

df_jour = pd.read_csv("features_journalier.csv", parse_dates=["date"])
df_jour = df_jour.sort_values("date").reset_index(drop=True)
df_jour["est_aout"] = (df_jour["date"].dt.month == 8).astype(int)
print(f"{len(df_jour)} jours charges | {df_jour['date'].min().date()} -> {df_jour['date'].max().date()}")
df_jour.head()


## 3. Constantes et split temporel (70/15/15, identique au pipeline local)

In [ ]:
TARGET = "volume_entrant_jour"
NOM_JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
EXOG_COLS = ["est_fin_de_mois", "est_fin_trimestre", "est_lundi", "est_vendredi", "est_aout"]
FEATURES_XGB = [
    "volume_lag_1j", "volume_lag_7j", "volume_lag_14j", "volume_lag_30j", "volume_moy_7j",
    "jour_semaine", "mois", "trimestre", "semaine_du_mois",
    "est_fin_de_mois", "est_fin_trimestre", "est_lundi", "est_vendredi", "est_aout",
]


def split_temporel(df, frac_train=0.70, frac_val=0.15):
    n = len(df)
    n_train = int(n * frac_train)
    n_val = int(n * frac_val)
    return df.iloc[:n_train].copy(), df.iloc[n_train:n_train + n_val].copy(), df.iloc[n_train + n_val:].copy()


def metriques(y_true, y_pred, label, dates=None):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, np.nan, y_true))) * 100
    print(f"  [{label}] MAE={mae:.1f}  RMSE={rmse:.1f}  MAPE={mape:.1f}%")
    if dates is not None:
        dow = pd.to_datetime(dates).dt.dayofweek
        mask = dow < 5
        if mask.sum() > 0:
            yt, yp = np.asarray(y_true)[mask], np.asarray(y_pred)[mask]
            mape_ouvre = np.mean(np.abs((yt - yp) / np.where(yt == 0, np.nan, yt))) * 100
            print(f"           MAPE jours ouvres = {mape_ouvre:.1f}%")
    return mae, rmse, mape


df_train, df_val, df_test = split_temporel(df_jour)
print(f"Split : train={len(df_train)}  val={len(df_val)}  test={len(df_test)}")


## 4. Modele Prophet (grille sur validation)

In [ ]:
def ajouter_regresseurs_prophet(df_in):
    out = df_in.copy()
    d = pd.to_datetime(out["ds"])
    out["est_fin_trimestre_reg"] = d.dt.month.isin([3, 6, 9, 12]).astype(int)
    out["est_fin_mois_reg"] = (d.dt.day >= 25).astype(int)
    out["est_aout_reg"] = (d.dt.month == 8).astype(int)
    return out


def entrainer_prophet(df_train_, changepoint_prior_scale, seasonality_mode):
    df_p = df_train_[["date", TARGET]].rename(columns={"date": "ds", TARGET: "y"})
    df_p = ajouter_regresseurs_prophet(df_p)
    model = Prophet(
        yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
        seasonality_mode=seasonality_mode, changepoint_prior_scale=changepoint_prior_scale,
    )
    for reg in ["est_fin_trimestre_reg", "est_fin_mois_reg", "est_aout_reg"]:
        model.add_regressor(reg)
    model.fit(df_p)
    return model


def predire_prophet(model, dates):
    df_f = pd.DataFrame({"ds": dates})
    df_f = ajouter_regresseurs_prophet(df_f)
    forecast = model.predict(df_f)
    return forecast["yhat"].values, forecast["yhat_lower"].values, forecast["yhat_upper"].values


print("Recherche d'hyperparametres Prophet (grille sur validation)...")
grille_prophet = [(cps, mode) for cps in [0.01, 0.05, 0.1, 0.5] for mode in ["additive", "multiplicative"]]
meilleur_prophet = None
for cps, mode in grille_prophet:
    model = entrainer_prophet(df_train, cps, mode)
    pred_val, _, _ = predire_prophet(model, df_val["date"])
    mae_val = mean_absolute_error(df_val[TARGET].values, pred_val)
    print(f"  changepoint_prior_scale={cps:<5} seasonality_mode={mode:<14} -> MAE val={mae_val:.1f}")
    if meilleur_prophet is None or mae_val < meilleur_prophet[0]:
        meilleur_prophet = (mae_val, cps, mode)

best_cps, best_mode = meilleur_prophet[1], meilleur_prophet[2]
print(f"\n-> Meilleure config Prophet : changepoint_prior_scale={best_cps}, seasonality_mode={best_mode}")

model_prophet_final = entrainer_prophet(df_train, best_cps, best_mode)
pred_test_p, _, _ = predire_prophet(model_prophet_final, df_test["date"])
mae_test_p, rmse_test_p, mape_test_p = metriques(df_test[TARGET].values, pred_test_p, "Prophet - Test", df_test["date"])


## 5. Modele SARIMA (avec regresseurs exogenes calendaires)

In [ ]:
print("Recherche d'hyperparametres SARIMA (grille sur validation, avec exogenes)...")
serie_train = df_train.set_index("date")[TARGET].asfreq("D").interpolate()
exog_train = df_train.set_index("date")[EXOG_COLS].asfreq("D").ffill()
exog_val = df_val[EXOG_COLS]

grille_ordres = [(1, 1, 1), (2, 1, 1), (1, 1, 2), (2, 1, 2)]
seasonal_order = (1, 1, 1, 7)
meilleur_sarima = None
for order in grille_ordres:
    fit = SARIMAX(serie_train, exog=exog_train, order=order, seasonal_order=seasonal_order,
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    pred_val = fit.get_forecast(steps=len(df_val), exog=exog_val).predicted_mean.values
    mae_val = mean_absolute_error(df_val[TARGET].values, pred_val)
    print(f"  order={order} -> MAE val={mae_val:.1f}")
    if meilleur_sarima is None or mae_val < meilleur_sarima[0]:
        meilleur_sarima = (mae_val, order)

best_order = meilleur_sarima[1]
print(f"\n-> Meilleur ordre SARIMA : {best_order}")

fit_sarima = SARIMAX(serie_train, exog=exog_train, order=best_order, seasonal_order=seasonal_order,
                     enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
pred_test_sarima = fit_sarima.get_forecast(steps=len(df_test), exog=df_test[EXOG_COLS]).predicted_mean.values
mae_test_s, rmse_test_s, mape_test_s = metriques(df_test[TARGET].values, pred_test_sarima, "SARIMA - Test", df_test["date"])


## 6. Modele XGBoost (features de lag, prevision recursive)

In [ ]:
def rechercher_xgboost(df_train_, df_val_):
    X_train, y_train = df_train_[FEATURES_XGB], df_train_[TARGET]
    X_val, y_val = df_val_[FEATURES_XGB], df_val_[TARGET]
    grille = [{"max_depth": md_, "learning_rate": lr} for md_ in [3, 4, 6] for lr in [0.03, 0.05, 0.1]]
    meilleur, meilleure_mae = None, None
    for params in grille:
        m = XGBRegressor(n_estimators=500, subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                         reg_alpha=0.1, reg_lambda=1.0, random_state=42,
                         early_stopping_rounds=20, eval_metric="mae", verbosity=0, **params)
        m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        mae_v = mean_absolute_error(y_val, m.predict(X_val))
        if meilleure_mae is None or mae_v < meilleure_mae:
            meilleure_mae, meilleur = mae_v, m
    return meilleur


def predire_xgboost_recursif(model, historique, dates_futures):
    serie = historique.set_index("date")[TARGET].asfreq("D").interpolate()
    previsions = []
    for date_cible in dates_futures:
        row = {
            "jour_semaine": date_cible.dayofweek, "mois": date_cible.month, "trimestre": date_cible.quarter,
            "semaine_du_mois": ((date_cible.day - 1) // 7) + 1,
            "est_fin_de_mois": int(date_cible.day >= 25),
            "est_fin_trimestre": int(date_cible.month in [3, 6, 9, 12]),
            "est_lundi": int(date_cible.dayofweek == 0), "est_vendredi": int(date_cible.dayofweek == 4),
            "est_aout": int(date_cible.month == 8),
        }
        for lag in [1, 7, 14, 30]:
            date_lag = date_cible - timedelta(days=lag)
            row[f"volume_lag_{lag}j"] = serie.get(date_lag, serie.iloc[-1])
        row["volume_moy_7j"] = serie.iloc[-7:].mean() if len(serie) >= 7 else serie.mean()
        X_pred = pd.DataFrame([row])[FEATURES_XGB]
        pred = model.predict(X_pred)[0]
        previsions.append(pred)
        serie.loc[date_cible] = pred
    return np.array(previsions)


print("Recherche d'hyperparametres XGBoost (grille sur validation)...")
df_xgb = df_jour.dropna(subset=FEATURES_XGB + [TARGET]).copy()
df_train_x, df_val_x, df_test_x = split_temporel(df_xgb)
model_xgb = rechercher_xgboost(df_train_x, df_val_x)
print(f"-> Meilleurs hyperparametres : max_depth={model_xgb.max_depth}, learning_rate={model_xgb.learning_rate}")

pred_test_xgb = model_xgb.predict(df_test_x[FEATURES_XGB])
mae_test_x, rmse_test_x, mape_test_x = metriques(df_test_x[TARGET].values, pred_test_xgb, "XGBoost - Test", df_test_x["date"])


## 7. Comparaison et selection du modele

In [ ]:
comparaison = pd.DataFrame({
    "Modele": ["Prophet", "SARIMA", "XGBoost"],
    "MAE (test)": [mae_test_p, mae_test_s, mae_test_x],
    "RMSE (test)": [rmse_test_p, rmse_test_s, rmse_test_x],
    "MAPE (test)": [mape_test_p, mape_test_s, mape_test_x],
})
print(comparaison.to_string(index=False))

candidats = {"Prophet": mae_test_p, "SARIMA": mae_test_s, "XGBoost": mae_test_x}
modele_gagnant = min(candidats, key=candidats.get)
print(f"\nModele retenu (meilleur MAE test) : {modele_gagnant} (MAE={candidats[modele_gagnant]:.1f})")


## 8. Previsions futures J+1 a J+30 avec le modele retenu

In [ ]:
today = pd.Timestamp.today().normalize()
dates_candidates = pd.date_range(start=today + timedelta(days=1), periods=45, freq="D")
dates_futures = dates_candidates[dates_candidates.dayofweek < 5][:30]

if modele_gagnant == "Prophet":
    model_full = entrainer_prophet(df_jour, best_cps, best_mode)
    yhat, yhat_lower, yhat_upper = predire_prophet(model_full, pd.Series(dates_futures))
    modele_a_sauver = {"model": model_full, "type": "prophet",
                       "changepoint_prior_scale": best_cps, "seasonality_mode": best_mode}
elif modele_gagnant == "SARIMA":
    serie_full = df_jour.set_index("date")[TARGET].asfreq("D").interpolate()
    exog_full = df_jour.set_index("date")[EXOG_COLS].asfreq("D").ffill()
    exog_futur = pd.DataFrame({
        "est_fin_de_mois": [int(d.day >= 25) for d in dates_futures],
        "est_fin_trimestre": [int(d.month in [3, 6, 9, 12]) for d in dates_futures],
        "est_lundi": [int(d.dayofweek == 0) for d in dates_futures],
        "est_vendredi": [int(d.dayofweek == 4) for d in dates_futures],
        "est_aout": [int(d.month == 8) for d in dates_futures],
    }, index=dates_futures)
    fit_full = SARIMAX(serie_full, exog=exog_full, order=best_order, seasonal_order=seasonal_order,
                       enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    forecast_full = fit_full.get_forecast(steps=len(dates_futures), exog=exog_futur)
    yhat = forecast_full.predicted_mean.values
    ci = forecast_full.conf_int(alpha=0.2)
    yhat_lower, yhat_upper = ci.iloc[:, 0].values, ci.iloc[:, 1].values
    modele_a_sauver = {"model": fit_full, "type": "sarima", "order": best_order,
                       "seasonal_order": seasonal_order, "exog_cols": EXOG_COLS}
else:
    model_full = rechercher_xgboost(df_train_x, df_val_x)
    yhat = predire_xgboost_recursif(model_full, df_jour[["date", TARGET]], dates_futures)
    marge = rmse_test_x
    yhat_lower, yhat_upper = yhat - marge, yhat + marge
    modele_a_sauver = {"model": model_full, "type": "xgboost", "features": FEATURES_XGB}

df_prevision = pd.DataFrame({
    "date": dates_futures, "horizon_j": range(1, len(dates_futures) + 1),
    "jour_semaine": [NOM_JOURS[d.weekday()] for d in dates_futures],
    "volume_prevu": np.round(yhat).astype(int),
    "volume_prevu_min": np.round(yhat_lower).astype(int),
    "volume_prevu_max": np.round(yhat_upper).astype(int),
})
df_prevision.head(10)


## 9. Export des resultats

In [ ]:
modele_a_sauver.update({
    "mae_test_prophet": mae_test_p, "mae_test_sarima": mae_test_s, "mae_test_xgboost": mae_test_x,
    "modele_retenu": modele_gagnant, "prophet_disponible": True, "motif_echec_prophet": None,
})
joblib.dump(modele_a_sauver, "model_volumes.pkl")
df_prevision.to_csv("predictions_volumes_j30.csv", index=False, encoding="utf-8")

with open("metriques_bloc3a.txt", "w", encoding="utf-8") as f:
    f.write("BLOC 3A - COMPARAISON PROPHET vs SARIMA vs XGBOOST (execute sur Google Colab)\n")
    f.write("=" * 70 + "\n\n")
    f.write(comparaison.to_string(index=False) + "\n\n")
    f.write(f"Modele retenu : {modele_gagnant} (MAE test = {candidats[modele_gagnant]:.1f})\n")
    if modele_gagnant == "Prophet":
        f.write(f"Config Prophet : changepoint_prior_scale={best_cps}, seasonality_mode={best_mode}\n")
    elif modele_gagnant == "SARIMA":
        f.write(f"Ordre SARIMA : {best_order}, seasonal_order={seasonal_order}\n")
    else:
        f.write(f"XGBoost : max_depth={model_xgb.max_depth}, learning_rate={model_xgb.learning_rate}\n")

print("Fichiers generes : model_volumes.pkl, predictions_volumes_j30.csv, metriques_bloc3a.txt")


## 10. Telechargement

Execute la cellule ci-dessous : les 3 fichiers se telechargent automatiquement.
Place-les ensuite dans `mon_projet/colab_prophet/resultats/` sur la machine
locale et signale-le pour integration au pipeline (table `predictions_volumes_j30`
et `models/model_volumes.pkl`).

In [ ]:
from google.colab import files
files.download("model_volumes.pkl")
files.download("predictions_volumes_j30.csv")
files.download("metriques_bloc3a.txt")
